# Select prediction rounds and reopen a fitted pipeline

## Goal and setup

Keep completed and future matches in the usual season-publication format, select
the exact rounds you want, and predict using a saved model. This example creates
two tiny synthetic publications in a new demo folder. It leaves real
data/xDiyo_data untouched. Use the existing Python (misc314) kernel.

### 1. Load explicit prediction rounds; retain all history

For real use, replace the synthetic helper with Path("data/xDiyo_data") and your
published seasons/leagues. No special daily directory is needed. Default awarded
exclusion removes only explicit True flags and their event-linked rows. Missing
kickoffs in this example deliberately produce a loader warning and sort last.
Use a fresh output folder when rerunning this save demonstration.

In [1]:
from pathlib import Path
import os, sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
source_root = Path(os.environ.get("XDIYO_VERIFIED_SOURCE", str(root)))
sys.path[:0] = [str(source_root / "src"), str(source_root), str(root)]
from pathlib import Path
import numpy as np
import pandas as pd
from notebooks.helpers.prediction_publications import create_demo_publications
from xdiyo_analytics.data import load_prediction_fixtures
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import Stat, IsHome, RollingMean, evaluate_features
from xdiyo_analytics.labels import MatchTotal, create_labels
from xdiyo_analytics.datasets import assemble_dataset
from xdiyo_analytics.training import EstimatorAdapter, refit_model, save_model, load_model
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge

output = root / "experiment/prediction_persistence_demo/notebook_revised"
data_root = create_demo_publications(output / "synthetic_exports")
batch = load_prediction_fixtures(data_root, "26_27", tables=["matches", "statistics"],
    rounds={"Alpha": [5, 6], "Beta": [6, 7]}, verify_hashes=True)
assert len(batch.data.matches) == 14
assert len(batch.completed_matches) == 4
assert len(batch.fixtures) == 4
print(batch.fixtures[["source_league", "event_id", "round", "status"]])


  source_league             event_id  round      status
0         Alpha  9223372036854775911      6  notstarted
1          Beta  9223372036854775911      6  notstarted
2         Alpha  9223372036854775910      5  notstarted
3          Beta  9223372036854775913      7  notstarted


C:\Users\luisi\Documents\Programming\Python\xDiyo\.pytest_tmp\prediction_persistence_verification\source_revision_2\src\xdiyo_analytics\data\loading.py:272: UserWarning: 1 matches have missing or unreadable kickoff times; rows retained.
  loaded = load_season(root, stem, tables=tables,
C:\Users\luisi\Documents\Programming\Python\xDiyo\.pytest_tmp\prediction_persistence_verification\source_revision_2\src\xdiyo_analytics\data\loading.py:272: UserWarning: 1 matches have missing or unreadable kickoff times; rows retained.
  loaded = load_season(root, stem, tables=tables,


### 2. Build features on all history, then align selected fixtures

History retains early current-season completed matches. Match features and labels
are assembled with drop_missing_targets=False so upcoming rows keep unknown
outcomes. The final selection intersects the loaded batch: Alpha round 5 and Beta
round 6, in kickoff order. No rows or history are invented. Team-match layout is
also supported and retains home then away rows for each fixture.

In [2]:
history = build_team_history(batch.data)
corners = Stat("ALL", "Match overview", "cornerKicks")
features = evaluate_features(history, {
    "home": IsHome(), "recent": RollingMean(corners, window=2),
}, keyed=True)
labels = create_labels(history, {"corners": MatchTotal(corners)})
dataset = assemble_dataset(features, labels["corners"], layout="match", drop_missing_targets=False)
prediction_data = batch.align(dataset, rounds={"Alpha": 5, "Beta": 6})
assert len(dataset.X) == 14 and len(prediction_data.X) == 2
assert prediction_data.y.isna().all().all()
assert prediction_data.metadata.source_league.tolist() == ["Beta", "Alpha"]


### 3. Save the entire fitted pipeline and predict after loading

Only the four finished, labelled synthetic matches train this smoke model.
Imputation, scaling and Ridge are saved together. Reloaded predictions match the
original model without refitting, and the two future target labels remain missing.
No forecasting performance claim is made. The fixture display below is a retained
prediction view using partition="test"; unknown observed results are neutral.
The empty scoring population is kept separate from these prediction rows.

Joblib/pickle loading can execute Python. Load artifacts from a trusted producer
in a compatible environment. A new model directory is required; saves never
overwrite existing artifacts. Numerical experiment recovery still returns
model=None and does not implicitly load executable models.

In [3]:
training_rows = np.flatnonzero(
    dataset.metadata.status.eq("finished").to_numpy()
    & dataset.y.notna().all(axis=1).to_numpy()
)
def model_factory():
    return EstimatorAdapter(make_pipeline(
        SimpleImputer(keep_empty_features=True), StandardScaler(), Ridge(alpha=.2),
    ))
fitted = refit_model(dataset, model_factory, train_positions=training_rows)
expected = fitted.predict(prediction_data)
model_path = fitted.save(output / "ridge_model")
restored = load_model(model_path)
actual = restored.predict(prediction_data)
pd.testing.assert_frame_equal(actual["predict"], expected["predict"])
predictions = prediction_data.metadata[["source_league", "event_id", "round", "home_id", "away_id"]].copy()
predictions["predicted_corners"] = actual["predict"].iloc[:, 0]
predictions.to_csv(output / "predictions.csv", index=False)
print(predictions)

from xdiyo_analytics.analysis import PostTrainingAnalysis
from xdiyo_analytics.reporting import MatchResultReporter
evaluation = restored.evaluate(prediction_data, test_positions=np.arange(len(prediction_data.X)),
                              score_positions=[], same_dataset=False)
report = PostTrainingAnalysis({
    "Upcoming fixtures": MatchResultReporter(type="per_fold", partition="test", target="corners", tolerance=1.),
}, title="Synthetic selected prediction rounds").run(evaluation)
report.to_html(output / "predictions.html")
report.to_notebook(height=850)


  source_league             event_id  round           home_id  \
0          Beta  9223372036854775911      6  9007199254740995   
1         Alpha  9223372036854775910      5  9007199254740995   

            away_id  predicted_corners  
0  9007199254740997                8.0  
1  9007199254740997                8.0  


## Continue

The [guide](../docs/analytics/prediction_persistence.md) includes a complete native
NPZ/JSON serializer example. See the [reference](../docs/analytics/prediction_persistence_reference.md),
[coverage](../docs/analytics/prediction_persistence_documentation_checklist.md), and
[verification](../docs/analytics/prediction_persistence_check.json).
Fresh-kernel execution and saved iframe interactions are verified; live Jupyter
frontend trust/display remains unverified. Fourteen earlier notebooks are unchanged.